**Dữ liệu:** 

1. movies.txt
- **Schema**: MovieID, Title, Genres

2. ratings_1.txt, ratings_2.txt
- **Schema**: UserID, MovieID, Rating, Timestamp

3. users.txt
- **Schema**: UserID, Gender, Age, Occupation, Zip-code

4. occupation.txt 
- **Schema**: ID, Occupation

**Lưu ý:** Thực hiện các bài tập dưới đây sử dụng RDD.

In [ ]:
from pyspark import SparkContext
from datetime import datetime

sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")


def parse_movie(line):
    movie_id, title, genres = line.split(",", 2)
    return int(movie_id), title, genres.split("|")


def parse_rating(line):
    user_id, movie_id, rating, timestamp = line.split(",")
    return int(user_id), int(movie_id), float(rating), int(timestamp)


def parse_user(line):
    user_id, gender, age, occupation, zip_code = line.split(",")
    return int(user_id), gender, int(age), int(occupation), zip_code


def parse_occupation(line):
    occupation_id, occupation = line.split(",", 1)
    return int(occupation_id), occupation


def age_group(age):
    if age < 18:
        return "under_18"
    if age <= 24:
        return "18-24"
    if age <= 34:
        return "25-34"
    if age <= 44:
        return "35-44"
    return "45+"


def summarize(pairs_rdd):
    return pairs_rdd.reduceByKey(lambda left, right: (left[0] + right[0], left[1] + right[1])).mapValues(lambda value: (value[0] / value[1], value[1]))


movies = sc.textFile("data/movies.txt").filter(lambda line: line.strip()).map(parse_movie).cache()
ratings = sc.textFile("data/ratings_1.txt").union(sc.textFile("data/ratings_2.txt")).filter(lambda line: line.strip()).map(parse_rating).cache()
users = sc.textFile("data/users.txt").filter(lambda line: line.strip()).map(parse_user).cache()
occupations = sc.textFile("data/occupation.txt").filter(lambda line: line.strip()).map(parse_occupation).cache()

movie_titles = movies.map(lambda item: (item[0], item[1])).cache()
movie_genres = movies.flatMap(lambda item: [(item[0], genre) for genre in item[2]]).cache()
ratings_by_movie = ratings.map(lambda item: (item[1], item[2])).cache()
ratings_by_user = ratings.map(lambda item: (item[0], (item[1], item[2]))).cache()
user_genders = users.map(lambda item: (item[0], item[1])).cache()
user_age_groups = users.map(lambda item: (item[0], age_group(item[2]))).cache()
user_occupations = users.map(lambda item: (item[3], item[0])).join(occupations).map(lambda item: (item[1][0], item[1][1])).cache()


def year_from_timestamp(timestamp):
    return datetime.utcfromtimestamp(timestamp).year

**Bài 1**: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim
* **Mục tiêu**:
  * Tính điểm trung bình của mỗi phim.
  * Đếm tổng số lượt đánh giá.
  * Tìm phim có điểm trung bình cao nhất (chỉ xét những phim có ít nhất 50 lượt đánh giá).
* **Giải pháp**:
  * Bước 1: Đọc file movies.txt và tạo một map (MovieID → Title).
  * Bước 2: Đọc file ratings_1.txt và ratings_2.txt, map MovieID → (Rating, 1).
  * Bước 3: Reduce để tính tổng điểm và số lượt đánh giá.
  * Bước 4: Tính điểm trung bình, lọc ra phim có ít nhất 5 lượt đánh giá.
  * Bước 5: Tìm phim có điểm trung bình cao nhất.

Bài 2: Phân tích đánh giá theo thể loại
* Mục tiêu:
  * Tính điểm trung bình của từng thể loại phim.
* Giải pháp
  * Bước 1: Tạo map (MovieID → List of Genres).
  * Bước 2: Map từ MovieID → Rating → (Genre, Rating).
  * Bước 3: Tính trung bình điểm đánh giá cho từng thể loại.

Bài 3: Phân tích đánh giá theo giới tính
* Mục tiêu:
  * Tính điểm trung bình của mỗi phim theo giới tính.
* Giải pháp
  * Bước 1: Tạo map (UserID → Gender).
  * Bước 2: Join với ratings để thêm thông tin giới tính.
  * Bước 3: Tính trung bình rating cho mỗi phim theo từng giới tính.

Bài 4: Phân tích đánh giá theo nhóm tuổi
* Mục tiêu:
  * Phân loại người dùng theo nhóm tuổi và tính điểm trung bình của mỗi phim theo từng nhóm.
* Giải pháp
  * Bước 1: Tạo map (UserID → Age Group).
  * Bước 2: Join với ratings để thêm nhóm tuổi.
  * Bước 3: Tính trung bình điểm đánh giá theo nhóm tuổi.

Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng
* Mục tiêu:
  * Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.
* Giải pháp:
  * Tạo dictionary từ users.txt với mapping UserID → Occupation.
  * Với mỗi rating, gán thông tin Occupation theo UserID.
  * Phát hành cặp key-value với key là Occupation và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi Occupation, sau đó tính trung bình rating.

Bài 6: Phân Tích Đánh Giá Theo Thời Gian
* Mục tiêu:
  * Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.
* Giải pháp:
  * Đọc dữ liệu ratings (từ cả ratings_1.txt và ratings_2.txt).
  * Sử dụng hàm trợ giúp để chuyển đổi Timestamp (dạng Unix) thành năm (Year).
  * Với mỗi dòng ratings, phát hành cặp key-value với key là năm và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi năm, sau đó tính trung bình.